# Chapter 2 — Documents, Data Sources & Ingestion

> **Where this sits:** Chapter 1 ended with `Documents → Chunk → Embed`. This chapter is
> everything that happens *before* that first arrow. It is where most real-world RAG
> systems quietly fail — and where almost no tutorial spends time.

This chapter answers one question:

> **How do we convert messy enterprise data into clean, structured, retrievable knowledge
> before chunking and embeddings even begin?**

Most developers think RAG starts here:

```
Document  →  Embedding  →  Vector DB
```

That is incomplete. A production RAG system actually starts much earlier:

```
Raw Sources
     │
     ▼
Source Connectors
     │
     ▼
File / Record Acquisition
     │
     ▼
  Parsing
     │
     ▼
OCR / Layout Understanding
     │
     ▼
Structure Extraction
     │
     ▼
  Cleaning
     │
     ▼
 Normalization
     │
     ▼
Metadata Extraction
     │
     ▼
 Deduplication
     │
     ▼
  Validation
     │
     ▼
Canonical Documents
     │
     ▼
  Chunking
     │
     ▼
 Embeddings
```

**If this pipeline is bad, the best embedding model, vector database, reranker and LLM in
the world will not save your RAG system.**


## What you will be able to do by the end

1. Explain why PDF parsing is genuinely hard (a top-5 interview question)
2. Design a **canonical document schema** that decouples sources from downstream code
3. Parse PDF, DOCX, HTML, CSV and JSON — each with the right strategy
4. Clean text without destroying meaning (the over-cleaning trap)
5. Detect exact and near duplicates
6. Build incremental, idempotent ingestion with change and deletion detection
7. Attach metadata, provenance and ACLs so retrieval can filter and cite
8. Walk the full RAG debugging ladder from source data to generated answer


## Table of contents

| § | Topic |
|---|-------|
| 1 | Why ingestion matters so much |
| 2 | The canonical document pattern |
| 3 | Source connectors |
| 4 | PDF — the hard one |
| 5 | DOCX, HTML, Markdown |
| 6 | CSV, JSON, databases, APIs |
| 7 | Metadata |
| 8 | Cleaning & normalization |
| 9 | Deduplication & versioning |
| 10 | Incremental ingestion |
| 11 | Tables, images, and special document types |
| 12 | Security & access control |
| 13 | Production architecture |
| 14 | Validation, observability & testing |
| 15 | Interview preparation |

## Setup

In [ ]:
# `sys` lets us modify Python's module search path.
import sys

# Add the project root so `utils` is importable from this notebook.
sys.path.append("..")

# `Path` gives clean, cross-platform file paths.
from pathlib import Path

# Regular expressions - the workhorse of text cleaning.
import re

# Point at the shared sample-data folder used by every notebook.
DATA_DIR = Path("../assets/sample_data")

# Create it if this is the first time the notebook runs.
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Confirm where we are writing.
print("Sample data folder:", DATA_DIR.resolve())

### Optional dependencies

The parsing sections need a few libraries. Each cell checks availability and skips
gracefully, so you can work through the chapter even with a partial install.

```bash
pip install pymupdf python-docx beautifulsoup4 lxml
```

In [ ]:
# Define a small helper that tries to import a module and reports the result.
def try_import(module_name, friendly_name, install_hint):

    # Attempt the import inside a try block so a missing package is not fatal.
    try:
        module = __import__(module_name)

    # If the package is not installed, report it and return None.
    except ImportError:
        print(f"  [ ] {friendly_name:<16} missing  -  pip install {install_hint}")
        return None

    # On success, report availability and hand back the module object.
    print(f"  [x] {friendly_name:<16} available")
    return module


# Announce what we are checking.
print("Optional parsing libraries:")

# PyMuPDF was historically imported as `fitz`; recent versions prefer `pymupdf`
# and emit a deprecation warning for the old name. Try the modern name first,
# then fall back, so this notebook works on both old and new installs.
fitz = try_import("pymupdf", "PyMuPDF", "pymupdf") or try_import("fitz", "PyMuPDF (legacy)", "pymupdf")

# python-docx is imported as `docx` - reads Word files including their styles.
docx = try_import("docx", "python-docx", "python-docx")

# bs4 is BeautifulSoup - an HTML/XML parser with a forgiving API.
bs4 = try_import("bs4", "BeautifulSoup", "beautifulsoup4")

# 1. Why ingestion matters so much

## 1.1 A failure that has nothing to do with your retriever

Imagine a PDF contains:

```
Section 7.2 — Leave Policy

Employees with at least five years
of continuous service are entitled
to 30 days of annual leave.
```

But your PDF parser extracts this:

```
Page 18
Company Confidential

Employees with at least five years

Page 19
Company Confidential

of continuous service are entitled

Page 20
Company Confidential

to 30 days of annual leave.
```

Your downstream pipeline now sees garbage structure. Chunking might produce:

```
Chunk 1: Page 18 Company Confidential Employees with at least five years
Chunk 2: Page 19 Company Confidential of continuous service are entitled
Chunk 3: Page 20 Company Confidential to 30 days of annual leave
```

Now the retriever can never return the **condition** and the **answer** together. A user
asking *"Who gets 30 days of leave?"* gets Chunk 3 — and is told **every** employee gets
30 days.

Your embedding model was fine. Your vector DB was fine. Your LLM was fine. The failure
happened three stages before any of them ran.

## 1.2 The ceiling

$$\text{RAG Quality} \le \text{Quality of Available Parsed Knowledge}$$

Extending the model from Chapter 1:

$$Q_{\text{RAG}} \approx Q_{\text{ingestion}} \times Q_{\text{retrieval}} \times Q_{\text{generation}}$$

(Conceptual, not a formal law.) If $Q_{\text{ingestion}} = 0.4$, even a perfect retriever
and a perfect generator are capped at 0.4.

Stated plainly:

> **A retriever cannot retrieve information that ingestion failed to capture.**

## 1.3 What "ingestion" means precisely

**Ingestion** is the complete process of taking external data and transforming it into a
standardized representation suitable for indexing and retrieval:

```
External data → Acquire → Parse → Understand structure → Clean
              → Normalize → Add metadata → Validate → Store canonical form
```

And a distinction that trips people up:

$$\text{Ingestion} \neq \text{Chunking}$$

Chunking is one *later* stage of the ingestion/indexing pipeline. Keep them separate in
your head and in your code.

## 1.4 What can enter a RAG system

```
                        DATA SOURCES
                             │
        ┌────────────────────┼────────────────────┐
        ▼                    ▼                    ▼
    Documents            Databases             Services
        │                    │                    │
    PDF / DOCX           SQL / NoSQL             APIs
    PPT / TXT            Warehouses              SaaS
    HTML                 Graph DB                Web
    Markdown
```

Typical enterprise reality: PDFs, DOCX, PowerPoint, Excel, CSV, JSON, XML, HTML, Markdown,
plain text, relational databases, NoSQL, SharePoint, Confluence, Google Drive, OneDrive,
S3, websites, APIs, support tickets, Slack/Teams, email, source-code repos, scanned forms,
images, tables, knowledge graphs.

# 2. The canonical document pattern

## 2.1 The problem

Different loaders produce different shapes: PDF gives pages, DOCX gives styled paragraphs,
HTML gives a DOM, SQL gives rows, an API gives JSON.

**Do not let every downstream component understand all those formats.** That is an
N-formats × M-consumers explosion.

## 2.2 The fix

Convert everything into **one standard object** immediately after parsing:

```
PDF ──────┐
DOCX ─────┤
HTML ─────┤
SQL ──────┼──►  Canonical Document Schema  ──►  Same downstream pipeline
API ──────┤
Email ────┤
Wiki ─────┘
```

Everything downstream — chunker, embedder, indexer, retriever — expects exactly:

```
Document
  ├── document_id
  ├── text
  └── metadata
```

This one decision removes an enormous amount of complexity.

In [ ]:
# `dataclass` generates __init__, __repr__ and __eq__ from field declarations.
from dataclasses import dataclass, field

# Typing helpers make the schema self-documenting.
from typing import Dict, Any, Optional


# Define the canonical document representation used by the whole pipeline.
@dataclass
class Document:

    # A stable, source-derived identifier (never a fresh random UUID - see section 10).
    document_id: str

    # The cleaned, normalized text that will eventually be chunked and embedded.
    text: str

    # Everything ABOUT the content: source, page, dates, department, permissions.
    metadata: Dict[str, Any] = field(default_factory=dict)

    # The original extraction, kept so we can re-run cleaning without re-parsing.
    raw_text: Optional[str] = None

    # A content hash, used for deduplication and change detection.
    checksum: Optional[str] = None


# Create one normalized document object to see the shape in practice.
document = Document(
    document_id="sharepoint:employee_policy_001",
    text="Employees receive 24 annual leave days.",
    metadata={
        "source": "sharepoint",
        "file_name": "employee_policy.pdf",
        "title": "Employee Policy",
        "page": 10,
        "department": "HR",
        "country": "India",
        "effective_date": "2026-04-01",
        "access_group": "employees",
    },
)

# Print it to confirm the structure.
print(document)

## 2.3 Why `raw_text` and `clean_text` are separate fields

This is an excellent production pattern worth adopting from day one:

```json
{
  "raw_text":   "...original extraction...",
  "clean_text": "...normalized extraction..."
}
```

**Why?** Six months from now you will discover your cleaning rule was too aggressive. With
`raw_text` preserved you can simply:

```
raw_text  →  re-run cleaning v2
```

without re-fetching or re-parsing a single source file. Without it, you re-download
everything.

A related benefit: retrieval can use `clean_text` while **citations display the original
text** exactly as the user would see it in the source document.

# 3. Source connectors

Before parsing, you have to actually *acquire* the data. A **connector** answers one
question:

> *Where is the document stored and how do I fetch it?*

```
                    Source Layer

SharePoint    S3    Database    Website    API
     │        │         │          │        │
     └────────┴─────────┴──────────┴────────┘
                        │
                        ▼
                 Connector Layer
                        │
                        ▼
                   Raw Content
```

## 3.1 Connector responsibilities

A good connector handles: **authentication, pagination, file discovery, download,
rate limits, retries, incremental updates, deletion detection, metadata acquisition,
permissions.**

**Do not put any of this inside the parser.** Separation of responsibilities:

```
Connector  →  gets bytes / records
Parser     →  understands content
```

That boundary keeps both testable.

## 3.2 The source-of-truth problem

Suppose the same policy exists in SharePoint, Google Drive, Confluence and an email
attachment. **Which is authoritative?**

Without a source strategy, RAG retrieves conflicting versions:

```
Version A: Refund = 14 days
Version B: Refund = 30 days
```

The retriever returns both. The LLM has no way to know which to trust — and you get a
hallucination or a hedge.

Production systems must store: **source priority · effective date · version ·
publication date · status.** We cover this in section 9.

# 4. PDF — the hard one

PDF is probably the most troublesome enterprise format for RAG.

## 4.1 Why

**PDF is a presentation format, not a semantic document format.**

A human sees:

```
TITLE

Introduction

This regulation applies to...

Table 1
...
```

Internally the PDF may store:

```
character x=120 y=515
character x=131 y=515
character x=144 y=515
...
```

There may be **no true concept of**: paragraph · section · table · reading order.

Everything a parser gives you about structure is *inferred* from coordinates.

## 4.2 Three broad PDF types — identify before you extract

```
PDF
│
├── Digital / text-based      →  extract text directly
│
├── Scanned / image-based     →  needs OCR
│
└── Mixed / complex-layout    →  needs layout-aware parsing
```

### Digital PDF
Generated from Word, a government publication, a documentation export. Contains
selectable text. Libraries: **PyMuPDF, pdfplumber, pypdf, Unstructured, Docling, Marker.**

### Scanned PDF
Essentially `PDF └── Image of page`. There may be **zero** machine-readable text:

```
Scanned PDF → Page image → OCR → Characters → Words → Paragraphs
```

OCR = **Optical Character Recognition**.

### Complex-layout PDF

```
┌────────────────────────────────────┐
│ TITLE                              │
├──────────────────┬─────────────────┤
│ Column 1         │ Column 2        │
│ text...          │ text...         │
├──────────────────┴─────────────────┤
│            Table                   │
├────────────────────────────────────┤
│ Figure 2          Caption          │
└────────────────────────────────────┘
```

A naive parser produces `TITLE Column 1 Column 2 text text Table Figure...` — reading order
destroyed.

## 4.3 The reading-order problem, concretely

```
COLUMN A              COLUMN B

A1                    B1
A2                    B2
A3                    B3
```

| | Sequence |
|---|---|
| **Correct reading order** | A1 A2 A3 B1 B2 B3 |
| **Naive parser output** | A1 B1 A2 B2 A3 B3 |

Semantics destroyed. Layout-aware parsers use bounding-box coordinates
$(x_1, y_1, x_2, y_2)$ for each text block to reconstruct:

```
page
 ├── title
 ├── paragraph
 ├── table
 ├── figure
 └── footer
```

A layout parser identifies regions such as: Title · Heading · Paragraph · List · Table ·
Image · Caption · Header · Footer · Equation.

```
            PDF PAGE

┌──────────────────────────┐
│ [TITLE]                  │
├──────────────────────────┤
│ [PARAGRAPH]              │
│                          │
├──────────────────────────┤
│ [TABLE]                  │
│                          │
├──────────────────────────┤
│ [FIGURE]     [CAPTION]   │
└──────────────────────────┘
```

This makes later chunking dramatically smarter.

### Let's create a real PDF and extract it

We generate a small multi-page PDF so the extraction code below runs against something
real rather than a made-up string.

In [ ]:
# Only run this cell if PyMuPDF is installed.
if fitz is not None:

    # Define where the generated sample PDF will live.
    sample_pdf = DATA_DIR / "employee_policy.pdf"

    # Create a brand-new, empty PDF document in memory.
    doc = fitz.open()

    # The text content for each page of our sample document.
    pages_content = [
        ("Employee Policy 2026\n\nSection 7.1 - Annual Leave\n\n"
         "Employees receive 24 annual leave days every year."),
        ("Section 7.2 - Long Service\n\n"
         "Employees with at least five years of continuous service\n"
         "are entitled to 30 days of annual leave."),
        ("Section 7.3 - Carry Forward\n\n"
         "Employees can carry forward up to 10 unused leave days."),
    ]

    # Build each page one at a time.
    for page_index, body in enumerate(pages_content, start=1):

        # Add a new blank page to the document.
        page = doc.new_page()

        # Insert a repeated header - deliberately, so we can strip it later.
        page.insert_text((72, 50), "COMPANY CONFIDENTIAL", fontsize=9)

        # Insert the actual body text below the header.
        page.insert_text((72, 100), body, fontsize=11)

        # Insert a repeated footer with the page number - also deliberate noise.
        page.insert_text((72, 760), f"Page {page_index} of 3", fontsize=9)

    # Write the finished PDF to disk.
    doc.save(sample_pdf)

    # Release the file handle.
    doc.close()

    # Confirm creation.
    print("Created:", sample_pdf)

### Basic PDF extraction with PyMuPDF

Notice the **design choice** in the code below: we return a *list of pages*, not one joined
string.

In [ ]:
# Define a function for extracting text from a digital PDF, page by page.
def extract_pdf_pages(pdf_path):

    # Open the PDF document from disk.
    pdf_document = fitz.open(pdf_path)

    # Create an empty list to collect one record per page.
    page_texts = []

    # Iterate through every page index in the PDF.
    for page_number in range(len(pdf_document)):

        # Load the page object at this index.
        page = pdf_document.load_page(page_number)

        # Extract the plain text of the page in reading order.
        page_text = page.get_text("text")

        # Store the page number alongside its text, so provenance survives.
        page_texts.append({
            "page_number": page_number + 1,   # human-facing pages start at 1
            "text": page_text,
        })

    # Close the file handle to release the OS resource.
    pdf_document.close()

    # Return the list of per-page records.
    return page_texts


# Only run if PyMuPDF is available.
if fitz is not None:

    # Extract all pages from the sample PDF we just created.
    pages = extract_pdf_pages(sample_pdf)

    # Show each page's extracted text.
    for record in pages:
        print(f"--- page {record['page_number']} ---")
        print(record["text"])

### Why joining pages immediately is dangerous

**Bad:**

```python
full_text = "".join(all_pages)
```

You lose: **page boundaries · citations · headers · sections · references.**

**Better:**

```json
[
  {"page": 1, "text": "..."},
  {"page": 2, "text": "..."}
]
```

Page number is valuable metadata for **citations, debugging and source linking**. Once you
throw it away at stage one, no later stage can recover it. Preserve provenance all the way
through chunking.

### Notice the noise

Look at the extracted output above. Every page carries `COMPANY CONFIDENTIAL` and
`Page N of 3`. Multiply that by an 800-page manual and you have thousands of tokens of pure
noise competing for retrieval space. We remove it in section 8.

### OCR: the pipeline and its failure modes

For scanned PDFs:

```
Page image
    │
    ▼
Image preprocessing
    │
    ├── deskew
    ├── denoise
    ├── contrast
    └── orientation
    │
    ▼
   OCR
    │
    ▼
Text + bounding boxes
    │
    ▼
Layout analysis
    │
    ▼
Reading order
```

Common engines: **Tesseract, PaddleOCR, Google Document AI, Azure Document Intelligence,
AWS Textract.**

### OCR is imperfect — and in regulated domains that matters enormously

Source says `Regulation 2026/1847`. OCR might output `Regulation 2026/184I` or
`Regu1ation 2026/1847`.

Classic confusions: `0`↔`O` · `1`↔`l` · `5`↔`S` · `rn`↔`m`

In legal or regulatory RAG, one character can change the meaning entirely.

### OCR confidence — a production pattern

Most OCR engines return a per-word confidence $c_i \in [0,1]$. Compute the page average:

$$C_{\text{page}} = \frac{1}{N}\sum_{i=1}^{N} c_i$$

Then act on it:

```
if C_page < 0.70:
    flag page  ·  retry OCR  ·  use stronger OCR model  ·  send for manual review
```

In [ ]:
# Define a function that computes average OCR confidence for a page.
def average_ocr_confidence(word_confidences):

    # Guard against an empty page, which would divide by zero.
    if not word_confidences:
        return 0.0

    # Average confidence is the sum of scores divided by the word count.
    return sum(word_confidences) / len(word_confidences)


# Define a function that decides what to do with an OCR result.
def route_ocr_result(word_confidences, threshold=0.70):

    # Calculate the page-level confidence score.
    confidence = average_ocr_confidence(word_confidences)

    # Low confidence means the text is probably unreliable - escalate it.
    if confidence < threshold:
        return f"FLAG for review (confidence {confidence:.2f} < {threshold})"

    # High confidence means we can accept the extraction.
    return f"ACCEPT (confidence {confidence:.2f})"


# Simulate a clean scan where the OCR engine was very sure of every word.
print("clean scan :", route_ocr_result([0.99, 0.97, 0.98, 0.95, 0.99]))

# Simulate a poor scan - faded, skewed, or low resolution.
print("poor scan  :", route_ocr_result([0.55, 0.61, 0.48, 0.72, 0.50]))

# 5. DOCX, HTML, Markdown

## 5.1 DOCX — much easier, because structure is explicit

Unlike PDF, a Word file **stores document structure**: Heading 1, Heading 2, Paragraph,
Bullet, Table, Image, Hyperlink. That structure is extremely valuable — do not throw it away.

In [ ]:
# Only run this cell if python-docx is installed.
if docx is not None:

    # Import the Document class under a clear alias.
    from docx import Document as WordDocument

    # Define where the generated sample DOCX will live.
    sample_docx = DATA_DIR / "leave_policy.docx"

    # Create a new, empty Word document.
    word_doc = WordDocument()

    # Add a top-level heading (level 1 maps to the "Heading 1" style).
    word_doc.add_heading("Employee Leave Policy", level=1)

    # Add a sub-heading for the first section.
    word_doc.add_heading("Annual Leave", level=2)

    # Add a normal body paragraph under that sub-heading.
    word_doc.add_paragraph("Employees receive 24 annual leave days every year.")

    # Add a second sub-heading.
    word_doc.add_heading("Permanent Employees", level=2)

    # Add body text whose meaning depends entirely on the heading above it.
    word_doc.add_paragraph("Employees must submit the declaration annually.")

    # Save the document to disk.
    word_doc.save(sample_docx)

    # Confirm creation.
    print("Created:", sample_docx)

In [ ]:
# Define a function that extracts paragraphs AND their structural style.
def extract_docx_paragraphs(docx_path):

    # Import here so the function is self-contained.
    from docx import Document as WordDocument

    # Open the Word document.
    word_document = WordDocument(docx_path)

    # Create a list to hold one record per non-empty paragraph.
    paragraphs = []

    # Iterate through every paragraph in document order.
    for paragraph in word_document.paragraphs:

        # Strip surrounding whitespace from the paragraph text.
        text = paragraph.text.strip()

        # Skip paragraphs that are empty after stripping.
        if not text:
            continue

        # Read the style name - this is how we know a heading from body text.
        style_name = paragraph.style.name

        # Store text together with its structural role.
        paragraphs.append({"text": text, "style": style_name})

    # Return all extracted paragraph records.
    return paragraphs


# Only run if python-docx is available.
if docx is not None:

    # Extract structured paragraphs from the sample file.
    for item in extract_docx_paragraphs(sample_docx):
        print(f"  [{item['style']:<10}] {item['text']}")

### Why headings must be preserved

This is the payoff for keeping `style`. Consider a document containing:

```
Section: Contractors
Employees must...
```

and later:

```
Section: Permanent Employees
Employees must...
```

If headings are discarded, a chunk reads:

> "Employees must submit the declaration annually."

**Which employees?** The heading held the answer. A better chunk carries its heading down
with it:

```
Section: Permanent Employees

Employees must submit the declaration annually.
```

Chapter 3 (structure-aware chunking) builds directly on this.

In [ ]:
# Define a function that attaches the current heading trail to each paragraph.
def attach_heading_context(paragraphs):

    # Track the most recent heading at each level, e.g. {1: "Policy", 2: "Annual Leave"}.
    heading_stack = {}

    # Collect the enriched records here.
    enriched = []

    # Walk the paragraphs in document order so headings precede their content.
    for item in paragraphs:

        # A style like "Heading 2" tells us this paragraph is a heading.
        if item["style"].startswith("Heading"):

            # Pull the numeric level out of the style name.
            level = int(item["style"].split()[-1])

            # Record this heading at its level.
            heading_stack[level] = item["text"]

            # Any deeper headings are now stale, so drop them.
            heading_stack = {k: v for k, v in heading_stack.items() if k <= level}

            # Headings themselves are context, not content - move on.
            continue

        # For body text, build the breadcrumb from the current heading stack.
        breadcrumb = " > ".join(heading_stack[k] for k in sorted(heading_stack))

        # Store the paragraph together with the section it belongs to.
        enriched.append({"section": breadcrumb, "text": item["text"]})

    # Return the enriched, context-carrying paragraphs.
    return enriched


# Only run if python-docx is available.
if docx is not None:

    # Show how ambiguous body text becomes unambiguous once context is attached.
    for item in attach_heading_context(extract_docx_paragraphs(sample_docx)):
        print(f"  [{item['section']}]")
        print(f"    {item['text']}\n")

Notice the second paragraph. On its own it was ambiguous. With `Employee Leave Policy >
Permanent Employees` attached, it is now retrievable *and* unambiguous. That is what
preserving structure buys you.

## 5.2 HTML — structured, but noisy

A web page contains: navigation · header · sidebar · cookie banner · ads · footer ·
**main content** · related links · comments.

```
<html>
 ├── <head>
 └── <body>
      ├── <nav>
      ├── <aside>
      ├── <main>
      │    ├── <h1>
      │    ├── <p>
      │    └── <table>
      └── <footer>
```

Naively calling `BeautifulSoup(html).get_text()` retrieves an enormous amount of
irrelevant boilerplate that will compete with your real content at retrieval time.

For RAG you usually care about: **main/article · headings · paragraphs · tables · lists.**

In [ ]:
# A realistic web page: real content wrapped in navigation, ads and a footer.
sample_html = """
<html>
  <head><title>Refund Policy</title><style>.ad { color: red; }</style></head>
  <body>
    <nav><a href="/">Home</a> <a href="/about">About</a> <a href="/shop">Shop</a></nav>
    <aside class="ad">SPECIAL OFFER! Buy now and save 50%!</aside>
    <main>
      <h2>Refund Policy</h2>
      <p>Refunds are permitted within 30 days of purchase.</p>
      <h2>Exchange Policy</h2>
      <p>Exchanges are permitted within 60 days of purchase.</p>
    </main>
    <footer>Copyright 2026 Example Corp. All rights reserved.</footer>
    <script>trackAnalytics();</script>
  </body>
</html>
"""

# Only run if BeautifulSoup is installed.
if bs4 is not None:

    # Import the parser class.
    from bs4 import BeautifulSoup

    # Parse the page once so we can compare two extraction strategies.
    soup_naive = BeautifulSoup(sample_html, "html.parser")

    # The naive approach: grab all text, boilerplate included.
    print("=== NAIVE get_text() ===")
    print(soup_naive.get_text(separator=" ", strip=True))

In [ ]:
# Define a function that extracts meaningful text and drops boilerplate.
def extract_html_text(html_content):

    # Import the parser class.
    from bs4 import BeautifulSoup

    # Parse the HTML into a navigable tree.
    soup = BeautifulSoup(html_content, "html.parser")

    # These tags essentially never contain answer-bearing content.
    noise_tags = ["script", "style", "nav", "footer", "aside", "header", "form"]

    # Remove each noise tag and everything inside it.
    for tag_name in noise_tags:
        for element in soup.find_all(tag_name):
            # decompose() deletes the element from the tree entirely.
            element.decompose()

    # Prefer <main> or <article> if the page marks its content region.
    content_root = soup.find("main") or soup.find("article") or soup

    # Extract the remaining visible text, one block per line.
    text = content_root.get_text(separator="\n", strip=True)

    # Return the cleaned text.
    return text


# Only run if BeautifulSoup is available.
if bs4 is not None:

    # Show the difference boilerplate removal makes.
    print("=== CLEANED ===")
    print(extract_html_text(sample_html))

### Preserving HTML structure beats flattening it

Given:

```html
<h2>Refund Policy</h2>
<p>Refunds are permitted within 30 days.</p>
```

Plain text is *fine*:

```
Refund Policy
Refunds are permitted within 30 days.
```

But a **structured** representation is better:

```json
{
  "heading": "Refund Policy",
  "paragraph": "Refunds are permitted within 30 days."
}
```

because it lets Chapter 3 do hierarchical chunking rather than guessing where sections start.

In [ ]:
# Define a function that pairs each heading with the paragraphs beneath it.
def extract_html_sections(html_content):

    # Import the parser class.
    from bs4 import BeautifulSoup

    # Parse the HTML.
    soup = BeautifulSoup(html_content, "html.parser")

    # Remove obvious boilerplate before looking for structure.
    for tag_name in ["script", "style", "nav", "footer", "aside"]:
        for element in soup.find_all(tag_name):
            element.decompose()

    # Collect one record per heading.
    sections = []

    # Walk every heading level in document order.
    for heading in soup.find_all(["h1", "h2", "h3"]):

        # Gather the paragraphs that follow this heading.
        paragraphs = []

        # find_next_siblings walks forward through the same level of the tree.
        for sibling in heading.find_next_siblings():

            # Stop as soon as we reach the next heading - that section is over.
            if sibling.name in ("h1", "h2", "h3"):
                break

            # Collect paragraph text.
            if sibling.name == "p":
                paragraphs.append(sibling.get_text(strip=True))

        # Store the heading together with its body text.
        sections.append({
            "heading": heading.get_text(strip=True),
            "paragraphs": paragraphs,
        })

    # Return the structured sections.
    return sections


# Only run if BeautifulSoup is available.
if bs4 is not None:

    # Show the structured output.
    for section in extract_html_sections(sample_html):
        print(f"  {section['heading']}")
        for paragraph in section["paragraphs"]:
            print(f"    - {paragraph}")

## 5.3 Markdown — the easiest format for RAG

Markdown is excellent for RAG because **structure is explicit in the text itself**:

```markdown
# Product Manual

## Installation
...

## Troubleshooting

### Network Issues
...
```

The hierarchy is immediately visible:

```
Product Manual
    │
    ├── Installation
    │
    └── Troubleshooting
            │
            └── Network Issues
```

**Preserve the heading levels** (`#`, `##`, `###`) rather than flattening everything — they
are free structure that Chapter 3 will use for header-based chunking.

# 6. CSV, JSON, databases and APIs

## 6.1 CSV — one question decides everything

CSV looks simple, but poses an important question:

> **What does one retrievable unit represent?**

Given:

```
customer_id,name,country,plan
101,Alice,India,Premium
102,Bob,Germany,Basic
```

You almost certainly do **not** want to concatenate:

```
101 Alice India Premium 102 Bob Germany Basic
```

Instead create **one semantic representation per row**:

```
Customer ID: 101
Name: Alice
Country: India
Plan: Premium
```

Notice what this buys you: the column *names* are now part of the embedded text, so a query
like *"which customers are on the Premium plan"* has lexical and semantic material to match
against.

In [ ]:
# csv is in the standard library - no install needed.
import csv

# Write a small CSV so the extraction code has real input.
sample_csv = DATA_DIR / "customers.csv"

# Open the file for writing with newline="" as the csv module requires.
with open(sample_csv, "w", newline="", encoding="utf-8") as file:

    # Create a writer bound to this file handle.
    writer = csv.writer(file)

    # Write the header row - these names become part of each embedded record.
    writer.writerow(["customer_id", "name", "country", "plan"])

    # Write the data rows.
    writer.writerow([101, "Alice", "India", "Premium"])
    writer.writerow([102, "Bob", "Germany", "Basic"])
    writer.writerow([103, "Chen", "Singapore", "Premium"])

# Confirm creation.
print("Created:", sample_csv)

In [ ]:
# Define a function that converts CSV rows into readable textual records.
def extract_csv_rows(csv_path):

    # Collect one serialized string per row.
    records = []

    # Open the CSV for reading with explicit UTF-8 encoding.
    with open(csv_path, "r", encoding="utf-8") as file:

        # DictReader uses the header row as keys, giving us column names.
        reader = csv.DictReader(file)

        # Iterate over every data row.
        for row in reader:

            # Build one "Column: value" line per field.
            fields = []

            # Walk the columns in their original order.
            for column_name, value in row.items():

                # Convert the pair into a readable line.
                fields.append(f"{column_name}: {value}")

            # Join the field lines into one record.
            records.append("\n".join(fields))

    # Return every serialized row.
    return records


# Serialize the sample CSV and show the result.
for record in extract_csv_rows(sample_csv):
    print(record)
    print("---")

## 6.2 JSON — flatten, but keep the hierarchy in the keys

JSON often contains nested structure:

```json
{
  "product": {
    "id": 143,
    "name": "Motor A",
    "specifications": { "voltage": "240V", "power": "5kW" }
  }
}
```

Flattening blindly to `product id 143 name Motor A specifications voltage 240V...` may be
fine for small records, but complex JSON needs the hierarchy preserved **in the key names**.

In [ ]:
# Define a function that flattens a nested JSON dictionary into flat dotted keys.
def flatten_json(data, parent_key=""):

    # Collect the flattened key/value pairs here.
    flattened = {}

    # Iterate through every key/value pair at this level.
    for key, value in data.items():

        # Build the full hierarchical key, e.g. "product.specifications.voltage".
        new_key = f"{parent_key}.{key}" if parent_key else key

        # If the value is another dictionary, we must go deeper.
        if isinstance(value, dict):

            # Recursively flatten the nested dictionary, passing the key prefix down.
            nested_values = flatten_json(value, new_key)

            # Merge the nested results into our output.
            flattened.update(nested_values)

        # Otherwise this is a leaf value we can store directly.
        else:

            # Store the value under its full hierarchical key.
            flattened[new_key] = value

    # Return the flattened dictionary.
    return flattened


# A nested product record to flatten.
product_json = {
    "product": {
        "id": 143,
        "name": "Motor A",
        "specifications": {"voltage": "240V", "power": "5kW"},
    }
}

# Flatten it.
flat = flatten_json(product_json)

# Show the flattened keys - note how hierarchy survives in the key names.
for key, value in flat.items():
    print(f"  {key}: {value}")

# Serialize into the text that would actually be embedded.
print("\nSerialized for embedding:")
print("\n".join(f"{k}: {v}" for k, v in flat.items()))

## 6.3 Should *all* JSON be embedded? No.

This is an important architectural judgement.

Imagine a million transactional records with `order_id, customer_id, amount, date`.
Embedding every record is **not** the right architecture.

For a precise structured query like *"What were total sales in Germany last month?"* you
want:

```
Question → Text-to-SQL → Database → Result → LLM
```

not semantic retrieval over serialized rows. A vector search will never compute a `SUM`.

This gives a key system-design principle:

> **Choose retrieval based on the structure and semantics of the data.**

```
DATA
│
├── Unstructured        → PDFs, emails, text, documents       → vector / hybrid
│
├── Semi-structured     → JSON, XML, HTML                     → depends on use
│
└── Structured          → SQL, warehouses, tables             → SQL / text-to-SQL
```

## 6.4 Databases — two architectures

**Architecture A — index database records**

```
Database → Extract rows → Embed → Vector DB
```

Good for *semantic* lookup ("motors suitable for high-vibration environments").

**Architecture B — query the database live**

```
Question → Generate SQL → Database → Current result
```

Good for: aggregations · current transactions · exact filtering · counts · sums · joins.

Production systems often combine both. Either way, when you serialize a row, **preserve the
original IDs as metadata**:

```json
{
  "text": "Product Name: Servo Motor X12\nCategory: Industrial Motor\n...",
  "metadata": { "product_id": 1725, "source_table": "products" }
}
```

because retrieval may later need to return the original database record.

## 6.5 APIs

You periodically fetch and convert responses into canonical documents. The concerns are:
**authentication · pagination · rate limiting · retries · timeouts · versioning ·
`updated_at` fields · deletion detection.**

# 7. Metadata

Metadata is information **about** the content. It is one of the most valuable — and most
frequently discarded — parts of RAG.

## 7.1 Why it matters: a precision disaster

User asks: *"What is the parental leave policy for Germany?"*

Pure vector search happily retrieves semantically similar policies from **India, Germany,
USA and UK** — they all read almost identically. Your top-3 may contain zero German content.

With metadata filtering:

```
Query
  │
  ▼
Country = Germany filter
  │
  ▼
Vector similarity
  │
  ▼
Top K
```

This **dramatically improves precision** for near-zero cost. Chapter 12 covers filtered
retrieval in depth; the point here is that you can only filter on metadata you **captured
during ingestion**.

## 7.2 Metadata categories

```
METADATA
│
├── Source Metadata
│      ├── file name
│      ├── URL
│      └── connector
│
├── Structural Metadata
│      ├── page
│      ├── heading
│      └── section
│
├── Business Metadata
│      ├── country
│      ├── product
│      └── department
│
├── Temporal Metadata
│      ├── created_at
│      ├── updated_at
│      └── effective_date
│
├── Security Metadata
│      ├── user_id
│      ├── role
│      └── access_group
│
└── Provenance Metadata
       ├── source_id
       ├── version
       └── checksum
```

## 7.3 Where metadata comes from

```
File system          Document              Connector
 ├── file name        ├── title             ├── permissions
 ├── path             ├── author            ├── source ID
 └── modified date    ├── effective date    └── URL
                      └── jurisdiction
```

## 7.4 Prefer deterministic metadata over LLM guessing

If the SharePoint API already returns `lastModifiedDateTime`, `createdBy`, `id`, `webUrl`
— **do not ask an LLM to infer them.**

```
Authoritative metadata source   >   LLM guessing
```

This improves accuracy and cuts cost.

LLM extraction *is* useful for content-level properties an API cannot know:

```
Document text → LLM extraction → jurisdiction = Germany
                                 document_type = regulation
                                 topic = battery recycling
```

But validate those outputs if they drive high-stakes filtering — and:

> ### Never derive security permissions from an LLM.

# 8. Cleaning & normalization

After extraction, raw text contains noise. Our sample PDF had `COMPANY CONFIDENTIAL` and
`Page N of 3` on every page. In an 800-page manual that is 1,600 lines of pure noise
consuming embeddings, retrieval slots and context tokens.

## 8.1 Common cleaning operations

```
Remove repeated headers        Normalize whitespace        Normalize Unicode
Remove repeated footers        Fix broken line breaks      Remove duplicated paragraphs
Remove page numbers            Remove control characters   Repair hyphenated words
                               Remove navigation boilerplate
```

**But cleaning must be conservative. Do not destroy meaningful text.**

## 8.2 Whitespace normalization

Extraction gives:

```
Employees      receive

24 annual
leave days.
```

You want: `Employees receive 24 annual leave days.`

The naive `re.sub(r"\s+", " ", text)` collapses *everything* — including the paragraph
breaks that carry structure. Two functions, two purposes:

In [ ]:
# Define an aggressive normalizer that collapses ALL whitespace into single spaces.
def normalize_whitespace(text):

    # \s+ matches any run of spaces, tabs and newlines; replace each run with one space.
    cleaned_text = re.sub(r"\s+", " ", text)

    # Remove leading and trailing whitespace.
    cleaned_text = cleaned_text.strip()

    # Return the fully flattened text.
    return cleaned_text


# Define a gentler normalizer that cleans within lines but keeps paragraph breaks.
def normalize_paragraph_whitespace(text):

    # Split the text into individual lines.
    lines = text.splitlines()

    # Collect the cleaned lines here.
    cleaned_lines = []

    # Process every line independently.
    for line in lines:

        # Collapse repeated spaces and tabs INSIDE the line (but not newlines).
        cleaned_line = re.sub(r"[ \t]+", " ", line)

        # Remove whitespace at the start and end of the line.
        cleaned_line = cleaned_line.strip()

        # Keep the cleaned line, including empty ones for now.
        cleaned_lines.append(cleaned_line)

    # Rejoin the lines with newlines.
    cleaned_text = "\n".join(cleaned_lines)

    # Collapse three-or-more blank lines down to a single paragraph break.
    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)

    # Return the text with paragraph structure intact.
    return cleaned_text


# A messy input with both intra-line noise and meaningful paragraph breaks.
messy = "Employees      receive\n\n\n\n24 annual\nleave days.\n\n\nSection 8 follows."

# The aggressive version destroys paragraph structure.
print("AGGRESSIVE:")
print(repr(normalize_whitespace(messy)))

# The gentle version preserves it.
print("\nPARAGRAPH-PRESERVING:")
print(repr(normalize_paragraph_whitespace(messy)))

## 8.3 The broken line-wrap problem

PDF extraction often gives:

```
Employees with at least five years of
continuous service are entitled to
thirty days of annual leave.
```

Those newlines are **visual formatting, not semantic paragraph breaks**. You want them
joined.

**But consider:**

```
1. Employees must...
2. Contractors must...
```

Blindly joining all lines destroys the list. So line repair should be layout-aware, or at
minimum should recognise list markers.

## 8.4 Hyphenation

PDF line-breaks words: `The organization must imple-\nment security controls.`

You want `implement`. But be careful — real hyphenated words like `risk-based` and
`long-term` **must stay hyphenated**.

In [ ]:
# Define a function that repairs hyphens introduced by line wrapping.
def repair_hyphenation(text):

    # Match: word char, hyphen, newline, word char - i.e. a word split across lines.
    # Replace with the two word chars joined, dropping the hyphen and newline.
    repaired_text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Return the repaired text.
    return repaired_text


# A word genuinely split by line wrapping.
wrapped = "The organization must imple-\nment security controls."

# A real hyphenated compound that happens to sit at a line end.
real_compound = "We apply a risk-\nbased approach."

# The first case is repaired correctly.
print("wrapped word :", repr(repair_hyphenation(wrapped)))

# The second case is repaired INCORRECTLY - "risk-based" became "riskbased".
print("real compound:", repr(repair_hyphenation(real_compound)))

> **Read that second output.** `risk-based` silently became `riskbased`. This is exactly
> why cleaning rules need a dictionary check, a hyphenated-term allowlist, or layout
> information about whether the line actually wrapped. A regex alone is not enough — and
> shipping one without knowing this is how subtle corruption enters a corpus.

## 8.5 Unicode normalization

You may encounter `é` and `é` — visually identical, but different Unicode byte sequences
(one precomposed, one an `e` plus a combining accent). Normalizing them makes deduplication
and exact matching work.

In [ ]:
# unicodedata provides the standard normalization forms.
import unicodedata


# Define a function that normalizes Unicode into canonical composed form.
def normalize_unicode(text):

    # NFC composes characters like "e + combining acute" into the single character "e-acute".
    normalized_text = unicodedata.normalize("NFC", text)

    # Return the normalized text.
    return normalized_text


# Build the same visible word two different ways.
precomposed = "café"              # single code point U+00E9
decomposed = "café"              # "e" followed by combining acute U+0301

# They look identical when printed.
print("look the same?      ", precomposed, decomposed)

# But Python considers them different strings.
print("equal before NFC?   ", precomposed == decomposed)

# After normalization they become genuinely equal.
print("equal after NFC?    ", normalize_unicode(precomposed) == normalize_unicode(decomposed))

## 8.6 Header / footer removal by frequency

We can detect boilerplate statistically. For a line $l$ across $N$ pages:

$$f(l) = \frac{\#\text{pages containing } l}{N}$$

If $f(l) > 0.8$ **and** the line sits near the top or bottom of the page, it is very likely
a header or footer.

In [ ]:
# Counter counts occurrences of hashable items.
from collections import Counter


# Define a function that detects lines repeated across many pages.
def detect_repeated_lines(page_texts, threshold=0.8):

    # Count how many PAGES each distinct line appears on.
    line_counter = Counter()

    # How many pages we are working with.
    total_pages = len(page_texts)

    # Guard against an empty document.
    if total_pages == 0:
        return set()

    # Process every page.
    for page_text in page_texts:

        # Use a SET so a line repeated twice on one page still counts once for that page.
        lines = {
            line.strip()
            for line in page_text.splitlines()
            if line.strip()
        }

        # Add this page's unique lines to the counter.
        line_counter.update(lines)

    # Collect the lines that qualify as boilerplate.
    repeated_lines = set()

    # Inspect each distinct line and its page count.
    for line, count in line_counter.items():

        # Fraction of pages on which this line appears.
        frequency = count / total_pages

        # Lines appearing on most pages are very likely headers or footers.
        if frequency >= threshold:
            repeated_lines.add(line)

    # Return the detected boilerplate lines.
    return repeated_lines


# Define a function that strips known boilerplate lines from a page.
def remove_repeated_lines(text, repeated_lines):

    # Split the text into lines.
    lines = text.splitlines()

    # Keep the lines that survive filtering.
    kept_lines = []

    # Examine every line.
    for line in lines:

        # Normalize for comparison against the boilerplate set.
        stripped_line = line.strip()

        # Drop the line if it matches known boilerplate.
        if stripped_line in repeated_lines:
            continue

        # Otherwise keep it, preserving original indentation.
        kept_lines.append(line)

    # Reassemble the cleaned page.
    return "\n".join(kept_lines)

In [ ]:
# Only run if we successfully extracted the sample PDF earlier.
if fitz is not None:

    # Pull just the text of each page.
    page_texts = [record["text"] for record in pages]

    # Detect which lines appear on most pages.
    boilerplate = detect_repeated_lines(page_texts, threshold=0.8)

    # Show what the heuristic flagged.
    print("Detected boilerplate:")
    for line in sorted(boilerplate):
        print("  -", repr(line))

    # Apply the removal to page 1 and show the before/after.
    print("\n--- page 1 BEFORE ---")
    print(page_texts[0])

    print("--- page 1 AFTER ---")
    print(remove_repeated_lines(page_texts[0], boilerplate))

Notice `Page 1 of 3` was **not** removed — it differs on every page, so its frequency is
only 1/3. Catching those needs a second rule (a regex for `Page \d+ of \d+`, or position on
the page). Real cleaners combine **frequency + page location + document semantics**, never
frequency alone.

## 8.7 Do not over-clean

This is the counterweight to everything above.

| Original | Over-cleaned | Damage |
|---|---|---|
| `Article 5(2)(a)` | `Article 5 2 a` | Legal citation semantics destroyed |
| `C++` | `C` | Different programming language |
| `ISO/IEC 27001:2022` | `ISO IEC 27001 2022` | Standard identifier mangled |
| `Straße` (German) | `Strasse` | Sometimes fine for search, wrong for legal names |

> ### Cleaning should remove noise, not information.

Boilerplate removal has the same trap. Removing `Confidential` from every page is fine. But
if every section repeats *"Applicable to hazardous materials only"*, a frequency-based
cleaner will delete a **critical legal condition**.

# 9. Deduplication & versioning

## 9.1 The problem

Enterprise file shares contain:

```
leave_policy.pdf
leave_policy_copy.pdf
leave_policy_final.pdf
leave_policy_final_v2.pdf
```

If all four are indexed, your top-5 retrieval becomes:

```
Top 5:  same policy · same policy · same policy · same policy · same policy
```

You have destroyed retrieval **diversity**. The user's follow-up question about a
*different* topic now has one slot instead of five.

## 9.2 Exact duplicates — hashing

For normalized text $x$, compute $h = \text{SHA256}(x)$. Identical hashes imply identical
normalized content.

In [ ]:
# hashlib provides cryptographic hash functions.
import hashlib


# Define a function that computes a stable content checksum.
def calculate_text_hash(text):

    # Encode the string into UTF-8 bytes - hash functions operate on bytes.
    text_bytes = text.encode("utf-8")

    # Compute SHA-256 and render it as a hexadecimal string.
    hash_value = hashlib.sha256(text_bytes).hexdigest()

    # Return the checksum.
    return hash_value


# Two documents with identical content.
doc_a = "Employees receive 24 days annual leave."
doc_b = "Employees receive 24 days annual leave."

# One document with a tiny wording change.
doc_c = "Employees receive 24 days of annual leave."

# Identical text produces identical hashes.
print("A:", calculate_text_hash(doc_a)[:16])
print("B:", calculate_text_hash(doc_b)[:16], " <- same as A")

# A single added word produces a completely different hash.
print("C:", calculate_text_hash(doc_c)[:16], " <- totally different")

## 9.3 Near duplicates — hashing is not enough

Note document C above. Adding the word "of" changed the hash completely — yet to a human
these are the same policy. Hashes catch **exact** duplicates only.

| Type | Example |
|---|---|
| Exact duplicate | `Employees receive 24 days annual leave.` |
| Near duplicate | `Employees receive 24 days **of** annual leave.` |
| Near duplicate | `Employees receive **twenty-four** days of annual leave.` |

Near-duplicate detection uses: **MinHash · SimHash · character similarity · embedding
similarity · Jaccard similarity.**

### Jaccard similarity

For token sets $A$ and $B$:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

*Example:*

```
A = {employee, receives, 24, annual, leave}
B = {employee, receives, 24, days, annual, leave}
```

Intersection = 5, union = 6, so $J = 5/6 = 0.833$ — likely a near duplicate.

In [ ]:
# Define a function for token-based Jaccard similarity.
def jaccard_similarity(text_a, text_b):

    # Lowercase and split into a SET of unique word tokens.
    tokens_a = set(text_a.lower().split())

    # Do the same for the second text.
    tokens_b = set(text_b.lower().split())

    # Tokens present in BOTH texts.
    intersection = tokens_a.intersection(tokens_b)

    # All distinct tokens across both texts.
    union = tokens_a.union(tokens_b)

    # Two empty strings are trivially identical - avoid dividing by zero.
    if not union:
        return 1.0

    # Jaccard = shared tokens / total distinct tokens.
    score = len(intersection) / len(union)

    # Return the similarity in the range [0, 1].
    return score


# Compare the near-duplicate pair that defeated hashing.
print(f"A vs C : {jaccard_similarity(doc_a, doc_c):.3f}  (near duplicate)")

# Compare against genuinely unrelated content.
print(f"A vs X : {jaccard_similarity(doc_a, 'The headquarters are in Munich.'):.3f}  (unrelated)")

### The scale problem

Do **not** compare every document with every other document. With $N$ documents that is:

$$\frac{N(N-1)}{2} \quad\text{comparisons} \quad\Rightarrow\quad O(N^2)$$

At $N = 1{,}000{,}000$ that is roughly $5 \times 10^{11}$ comparisons — impossible.

At scale, use **LSH / MinHash**, which bucket similar documents together so you only
compare within buckets. Chapter 8 covers LSH properly.

## 9.4 Versioning

Documents change:

```
Leave Policy v1  —  effective 2025
Leave Policy v2  —  effective 2026
```

Keep both? **It depends on the product requirement:**

| Requirement | Strategy |
|---|---|
| Current-policy QA | Keep v2 active, archive v1 |
| Legal / regulatory history | Keep all versions with full temporal metadata |

```json
{
  "version": "2.0",
  "effective_from": "2026-04-01",
  "effective_to": null,
  "status": "active"
}
```

### Effective date ≠ publication date

Critical in legal and regulatory RAG. A document can be **published January 2026** but
**effective July 2026**.

If a user asks *"What regulation applied in March 2026?"*, publication date alone gives the
wrong answer. Preserve: `publication_date · effective_date · repeal_date · version · status`.

### The amendment problem

```
Regulation A, Article 7:  Limit = 10 mg

Regulation B:  Article 7 of Regulation A is amended:  10 mg → 5 mg
```

Index both naively and the retriever returns **both 10 mg and 5 mg**. The LLM sees
contradictory facts with no way to resolve them.

Ingestion may need explicit legal relationships:

```
original regulation
        │
        ▼
    amended by
        │
        ▼
new effective provision
```

This illustrates that **domain-specific ingestion can matter as much as retrieval**.

# 10. Incremental ingestion

## 10.1 The naive approach is wasteful

```
Every night:
  download 1,000,000 docs
  re-parse everything
  re-embed everything
  re-index everything
```

If only 10,000 documents change per day, you are doing **99% unnecessary work**:

| | Documents/day |
|---|---|
| Full reprocessing | 1,000,000 |
| Incremental | 10,000 |
| **Reduction** | **99%** |

## 10.2 The incremental flow

```
Existing docs
     │
     ▼
Check source
     │
     ▼
New? Modified? Deleted?
     │
     ├── unchanged  →  do nothing
     ├── new        →  ingest
     ├── modified   →  reprocess
     └── deleted    →  remove
```

## 10.3 Change detection

| Technique | Check |
|---|---|
| Modified timestamp | `source_updated_at > last_sync` |
| ETag | HTTP / S3 / object-storage identifier |
| Content hash | `hash(old) ≠ hash(new)` |

Content hash is the most reliable — timestamps can change when content did not (a file
touched by a backup job).

In [ ]:
# Define the states a document can be in between two sync runs.
def detect_change(stored_record, current_record):

    # A document we have never seen before is new.
    if stored_record is None:
        return "NEW"

    # If the content hash is unchanged, nothing needs reprocessing.
    if stored_record["content_hash"] == current_record["content_hash"]:
        return "UNCHANGED"

    # Same ID, different hash - the content was edited.
    return "MODIFIED"


# Simulate the state we persisted after the last sync run.
stored_state = {
    "sharepoint:001": {"content_hash": calculate_text_hash("Employees receive 24 days.")},
    "sharepoint:002": {"content_hash": calculate_text_hash("Headquarters in Munich.")},
    "sharepoint:003": {"content_hash": calculate_text_hash("Old policy text.")},
}

# Simulate what the source looks like now: 001 unchanged, 002 edited, 003 deleted, 004 new.
current_state = {
    "sharepoint:001": {"content_hash": calculate_text_hash("Employees receive 24 days.")},
    "sharepoint:002": {"content_hash": calculate_text_hash("Headquarters in Berlin.")},
    "sharepoint:004": {"content_hash": calculate_text_hash("New travel policy.")},
}

# Classify every document currently present at the source.
for document_id, record in current_state.items():
    status = detect_change(stored_state.get(document_id), record)
    print(f"  {document_id}: {status}")

# Anything we stored but no longer see at the source has been DELETED.
for document_id in stored_state:
    if document_id not in current_state:
        print(f"  {document_id}: DELETED  <- must be removed from the index")

## 10.4 Deletion handling — the most commonly forgotten step

Suppose a policy is removed from SharePoint. But your vector DB still contains its chunks.
**Users can now retrieve deleted content forever.**

That is a serious correctness *and* security problem — imagine the deleted document was
removed precisely because it contained an error, or because someone lost access.

You need:

```
source deletion
      ↓
find document_id
      ↓
delete associated chunks
      ↓
delete embeddings
      ↓
update metadata/status
```

## 10.5 Stable IDs — the prerequisite for everything above

**Bad:**

```python
document_id = str(uuid.uuid4())   # a fresh random ID on every ingestion run
```

Every sync creates duplicates, and you can never match a source document to what you
already indexed.

**Better:**

```python
document_id = f"{connector}:{source_native_id}"   # e.g. "sharepoint:01ABCD2394"
```

This lets you **update** rather than duplicate, and lets deletion detection work at all.

Child chunks then carry their parent:

```
Document  document_id = D123
    │
    ├── chunk D123_001
    ├── chunk D123_002
    ├── chunk D123_003
    └── chunk D123_004
```

Every chunk must retain `parent_document_id` so you can reconstruct provenance and delete
cleanly.

## 10.6 Idempotency

A production ingestion operation should be **idempotent**:

$$f(f(x)) = f(x)$$

Running it twice must not create duplicates.

```
Run ingestion      → Document indexed
Run same ingestion → Still ONE document
```

Not:

```
Run 1 → document
Run 2 → duplicate
Run 3 → duplicate
```

This matters because message queues deliver **at least once**, not exactly once. Your
worker *will* occasionally see the same event twice. Stable IDs + content hashes + **UPSERT**
semantics give you idempotency:

```
IF document exists:  UPDATE
ELSE:                INSERT
```

## 10.7 Atomic replacement

When reprocessing a changed document while users are querying:

**Bad sequence:**

```
delete old chunks
      ↓
30 seconds of processing
      ↓
insert new chunks
```

For 30 seconds the document **disappears** from search results.

**Better:**

```
build new version
      ↓
   validate
      ↓
switch active version
      ↓
delete old version later
```

# 11. Tables, images and special document types

## 11.1 Tables — one of the biggest challenges in RAG

```
Country     Leave
India       24
Germany     30
France      25
```

Plain extraction gives: `Country Leave India 24 Germany 30 France 25`

The column-to-value relationships are now **weak**. An embedding of that string may well
match a query about France with the number 24.

**Better: serialize row-wise, carrying the header into every row.**

In [ ]:
# Define a function that converts table rows into readable, self-describing text.
def serialize_table_rows(headers, rows):

    # Collect one string per row.
    serialized_rows = []

    # Process every data row.
    for row in rows:

        # Build "Header: value" pairs for this row.
        fields = []

        # zip pairs each header with the value in the matching column.
        for header, value in zip(headers, row):

            # Create a semantically explicit field representation.
            fields.append(f"{header}: {value}")

        # Join the fields of this row with a visual separator.
        serialized_rows.append(" | ".join(fields))

    # Return all serialized rows.
    return serialized_rows


# The table headers.
headers = ["Country", "Leave"]

# The table body.
rows = [["India", "24 days"], ["Germany", "30 days"], ["France", "25 days"]]

# Serialize and display - each row is now independently retrievable and unambiguous.
for line in serialize_table_rows(headers, rows):
    print(" ", line)

### But table RAG often needs more than serialization

Consider:

```
Region   Q1   Q2   Q3   Q4
APAC     12   17   19   21
EMEA      9   14   16   20
```

Question: *"Which region had the largest percentage increase from Q1 to Q4?"*

This requires **calculation**:

$$\frac{Q4 - Q1}{Q1} \times 100$$

Pure semantic retrieval is insufficient — no amount of embedding similarity performs
arithmetic. Better architecture:

```
Question
   ↓
Retrieve relevant table
   ↓
Structured table parser
   ↓
Python / SQL calculation
   ↓
LLM explanation
```

Chapter 31 (Table RAG) covers this.

## 11.2 Images and figures

A technical manual says: *"Refer to Figure 7 for wiring configuration."* And **Figure 7
contains the actual information.** Text-only extraction loses it entirely.

Multimodal ingestion:

```
Image
 │
 ├── OCR text
 ├── caption
 ├── image description
 └── visual embedding
```

A useful canonical representation:

```json
{
  "type": "figure",
  "page": 17,
  "caption": "Figure 7: Motor wiring configuration",
  "ocr_text": "...",
  "description": "Diagram showing...",
  "image_reference": "s3://..."
}
```

**Captions matter enormously.** Even if you cannot process the visual yet, preserving the
caption, page and nearby paragraph materially improves retrieval.

## 11.3 Other formats, briefly

| Format | Preserve |
|---|---|
| **PowerPoint** | slide number · title · bullets · **speaker notes** · tables |
| **Excel** | workbook · sheet · table region · headers · row IDs · formulas where important |
| **Email** | sender · recipient · subject · timestamp · **thread ID** · reply relationships · attachments |
| **Code** | file path · class · function · imports · comments · symbol boundaries |
| **Legal/regulatory** | title · identifier · jurisdiction · publication date · effective date · articles · annexes · amendments · repealed status |

### Why email threads matter

Without thread structure you retrieve:

> *"Yes, approved."*

with **no idea what was approved**. With thread context:

```
Subject: Budget approval

Previous message:
Can we approve the €50K project budget?

Reply:
Yes, approved.
```

The same logic applies to Slack, Teams and support tickets — tiny messages embedded
independently lose all context. This connects directly to Chapter 3's parent-child chunking.

### Do not chunk code like prose

Code has hard semantic boundaries (function, class, module) that prose does not. Splitting
a function in half produces two chunks that are each individually meaningless.

## 11.4 Language and encoding

Enterprise corpora are multilingual. Store `{"language": "de"}` because it lets you choose
a multilingual embedding model, route OCR differently, apply language-specific tokenizers,
translate the query, or filter documents.

**Encoding** matters too. Broken decoding produces `FranÃ§ais` instead of `Français` — and
that mangled text gets embedded, indexed and retrieved forever. Detect and normalize
encoding **before** indexing.

# 12. Security & access control

Extremely important for enterprise RAG, and a frequent interview topic.

## 12.1 The leak

Suppose an **HR Salary Document** should only be visible to HR managers. If you ingest it
without permissions:

```
Vector DB
    ↓
any employee query
    ↓
salary document retrieved
```

**That is a severe data leak.** And note *where* it happened: at ingestion, months before
anyone asked the question.

> ### Permissions must travel with the document.

## 12.2 Security metadata

```json
{
  "allowed_users": ["U123", "U991"],
  "allowed_groups": ["HR_MANAGERS"],
  "classification": "CONFIDENTIAL"
}
```

At retrieval time:

```
User
  ↓
determine authorization
  ↓
metadata security filter
  ↓
search allowed documents only
```

## 12.3 The rule that wins the interview

Security must **not** rely on:

> *"Tell the LLM not to reveal salary information."*

**Prompt-level instructions are not a security boundary.** They can be talked around, and
they fail silently. Retrieval must prevent unauthorized content from **entering the context
at all**.

## 12.4 ACL inheritance

```
Folder A
 └── permission: HR only
      ├── file1
      └── file2
```

Files inherit folder permissions. Your connector must resolve the **effective ACL**, not
just the file-local ACL — otherwise `file1` looks unrestricted and leaks.

## 12.5 PII

Ingestion encounters names, addresses, emails, employee IDs, bank accounts, medical data.
Depending on requirements you may: **retain with access controls · mask · redact ·
tokenize · exclude.**

> **Never blindly embed all enterprise data.**

In [ ]:
# Define a minimal redaction function for demonstration.
def redact_account_numbers(text):

    # \b\d{9,}\b matches a standalone run of 9 or more digits - a plausible account number.
    # In production use a proper PII detection library; this is illustrative only.
    return re.sub(r"\b\d{9,}\b", "[REDACTED]", text)


# A sentence containing sensitive data.
sensitive = "Employee John Doe has account number 123456789."

# Show the redacted output.
print(redact_account_numbers(sensitive))

> Whether redaction is *appropriate* depends on the use case. A payroll assistant needs the
> account number; a general employee chatbot must not see it. This is a product decision,
> not a technical default — and the same corpus may need both, which is why access-controlled
> retrieval usually beats destructive redaction.

## 12.6 Provenance

For every chunk you later retrieve, you must be able to answer: **"Where did this come from?"**

```
Chunk ID
   ↓
Parent Document ID
   ↓
Source File
   ↓
Page 17
   ↓
SharePoint URL
   ↓
Version 4
```

This enables **citations · auditing · debugging · compliance**.

### Data lineage

Lineage tracks *transformations*:

```
SharePoint PDF
     ↓
PyMuPDF extraction
     ↓
header removal v2
     ↓
semantic chunker v4
     ↓
embedding model X
     ↓
Milvus collection Y
```

When something goes wrong, lineage tells you **which stage** to fix.

# 13. Production architecture

## 13.1 The full picture

```
                        DATA SOURCES
                             │
        ┌────────────────────┼────────────────────┐
        │                    │                    │
   SharePoint/S3       Database/API              Web
        │                    │                    │
        └────────────────────┼────────────────────┘
                             ▼
                     CONNECTOR SERVICE
                             │
                             ▼
                      Raw Object Store
                             │
                             ▼
                     DOCUMENT ROUTER
                             │
            ┌────────────────┼────────────────┐
            ▼                ▼                ▼
           PDF             DOCX             HTML
            │                │                │
            ▼                ▼                ▼
       PDF Parser      DOCX Parser       DOM Parser
            │
            ├── text detected ──────┐
            │                       │
            └── scan → OCR ─────────┤
                                    ▼
                            STRUCTURE LAYER
                        headings/tables/figures
                                    │
                                    ▼
                               CLEANING
                                    │
                                    ▼
                            NORMALIZATION
                                    │
                                    ▼
                         METADATA ENRICHMENT
                                    │
                                    ▼
                            SECURITY / ACL
                                    │
                                    ▼
                            DEDUPLICATION
                                    │
                                    ▼
                              VALIDATION
                                    │
                                    ▼
                      CANONICAL DOCUMENT STORE
                                    │
                                    ▼
                                CHUNKING
```

## 13.2 Three-layer storage — adopt this

```
Layer 1 — RAW              Layer 2 — CANONICAL PARSED     Layer 3 — RETRIEVAL INDEX

original PDF               text                            chunks
original DOCX              layout                          embeddings
original JSON              tables                          BM25 index
original HTML              metadata                        metadata filters
                           provenance
```

This mirrors the **medallion architecture** (Bronze → Silver → Gold) used in data
engineering.

### Why keep raw files?

Six months from now you discover *"Parser v1 incorrectly extracted tables."* With originals
retained:

```
raw file → parser v2 → reprocess
```

Otherwise you re-download 10 million documents from SharePoint.

### Why keep canonical parsed documents outside the vector DB?

Suppose you change your **chunking strategy** (which you will — repeatedly). You want:

```
Canonical parsed document → new chunking → new embeddings
```

**without re-parsing every PDF.** Parsing is the expensive stage; chunking is cheap. Keeping
them separate saves enormous time and money.

## 13.3 Batch vs event-driven

| | Batch | Event-driven |
|---|---|---|
| Flow | Every night → scan sources → process changes | Document uploaded → event → queue → worker |
| Freshness | Hours | Near-real-time |
| Complexity | Low | Higher |

```
SharePoint / S3
      │
      ▼
File Created Event
      │
      ▼
 Message Queue          (Kafka · RabbitMQ · SQS · Azure Service Bus · Pub/Sub)
      │
      ▼
 Parser Worker
      │
      ▼
 Chunk Worker
      │
      ▼
Embedding Worker
      │
      ▼
  Vector DB
```

### Why queues?

50,000 PDFs arrive simultaneously.

```
Without queue:                With queue:

  50,000                        50,000 events
     │                                │
     ▼                                ▼
parser service                      Queue
     │                                │
     ▼                          ┌─────┼─────┐
    💥                          ▼     ▼     ▼
                               W1    W2    W3
```

The queue absorbs the burst.

### Backpressure

Let $\lambda$ = incoming docs/sec and $\mu$ = processing docs/sec. If $\lambda > \mu$,
queue depth grows without bound. This is a fundamental distributed-systems concern, not a
RAG-specific one — but it will be your production incident.

## 13.4 Error handling

**Bad:**

```
100,000 files → file #427 corrupt → ENTIRE JOB FAILS
```

**Better:**

```
100,000 files
     │
     ├── success  99,972
     ├── retry        16
     ├── failed       12
     └── continue
```

### Retry with exponential backoff

$$\text{delay}_n = \min(\text{delay}_{\max},\; \text{delay}_0 \times 2^n)$$

```
Retry 1 → 1s      Retry 2 → 2s      Retry 3 → 4s      Retry 4 → 8s
```

Add **jitter** (a small random offset) to prevent thousands of workers retrying in lockstep
and re-creating the thundering herd you were avoiding.

In [ ]:
# random provides the jitter.
import random


# Define a function that computes the wait time before retry attempt n.
def retry_delay(attempt, base_delay=1.0, max_delay=60.0, jitter=True):

    # Exponential growth: 1s, 2s, 4s, 8s, ... capped at max_delay.
    delay = min(max_delay, base_delay * (2 ** attempt))

    # Add up to 25% random jitter so concurrent workers do not retry in lockstep.
    if jitter:
        delay = delay * (1 + random.random() * 0.25)

    # Return the computed delay in seconds.
    return delay


# Show the backoff schedule for the first six attempts.
for attempt in range(6):
    print(f"  attempt {attempt}: wait {retry_delay(attempt, jitter=False):5.1f}s "
          f"(with jitter ~{retry_delay(attempt):5.1f}s)")

### Dead-letter queue

If a document repeatedly fails:

```
Parser → retry → retry → retry → Dead Letter Queue
```

The DLQ stores failures for human inspection. Typical reasons: password-protected PDF ·
corrupt file · unsupported format · OCR failure · parser crash · malformed JSON.

## 13.5 Ingestion status machine

```
DISCOVERED → DOWNLOADED → PARSED → CLEANED → VALIDATED → CHUNKED → EMBEDDED → INDEXED

Failures:  FAILED_DOWNLOAD · FAILED_PARSE · FAILED_OCR · FAILED_EMBED · FAILED_INDEX
```

This makes observability enormously easier — you can query "how many documents are stuck at
PARSED?" instead of guessing.

## 13.6 Cost

Parsing costs CPU, memory, OCR API calls, LLM metadata extraction, storage and network. At
scale, tiny per-document costs explode.

Suppose OCR costs \$0.005/page and documents average 10 pages. For one million scanned PDFs:

$$1{,}000{,}000 \times 10 \times 0.005 = \$50{,}000$$

**So do not OCR digital PDFs unnecessarily. Route intelligently:**

```
PDF
 │
 ▼
Can text be extracted?
 │
 ├── YES → digital parser   (nearly free)
 │
 └── NO  → OCR              (expensive)
```

instead of `Every PDF → expensive OCR`.

In [ ]:
# Define a function that decides whether a PDF page needs OCR.
def needs_ocr(page_text, minimum_characters=50):

    # An empty or near-empty text layer means the page is very likely a scan.
    return len(page_text.strip()) < minimum_characters


# Define the routing decision for a whole document.
def route_pdf(page_texts, scan_ratio_threshold=0.5):

    # Count how many pages appear to lack a usable text layer.
    scanned_pages = sum(1 for text in page_texts if needs_ocr(text))

    # Guard against an empty document.
    if not page_texts:
        return "REJECT (no pages)"

    # What fraction of the document looks scanned.
    ratio = scanned_pages / len(page_texts)

    # Mostly scanned - send the whole thing to OCR.
    if ratio > scan_ratio_threshold:
        return f"OCR (scanned ratio {ratio:.0%})"

    # Some pages scanned - a hybrid approach is cheapest.
    if ratio > 0:
        return f"HYBRID: digital parser + OCR for {scanned_pages} page(s)"

    # Fully digital - use the cheap path.
    return "DIGITAL PARSER (free)"


# A realistic page of extracted text (comfortably above the 50-character threshold).
digital_page = (
    "Employees receive 24 annual leave days every year. Requests must be "
    "submitted through the HR portal at least two weeks in advance."
)

# A fully digital document - every page has a healthy text layer.
print("digital doc :", route_pdf([digital_page] * 3))

# A fully scanned document - the text layer is essentially empty on every page.
print("scanned doc :", route_pdf(["", "  ", "\n"]))

# A mixed document - two digital pages plus one scanned insert.
print("mixed doc   :", route_pdf([digital_page, "", digital_page]))

# 14. Validation, observability & testing

## 14.1 Validate before chunking

Possible checks: **text not empty · minimum content length · valid encoding · language
detected · required metadata exists · parser quality acceptable · OCR confidence acceptable
· duplicate check passed.**

In [ ]:
# Define a function that validates whether extracted text is usable.
def validate_extracted_text(text, minimum_characters=50):

    # Reject a missing value outright.
    if text is None:
        return False, "text is None"

    # Strip whitespace before measuring real content.
    stripped_text = text.strip()

    # Reject whitespace-only extractions.
    if not stripped_text:
        return False, "text is empty"

    # Reject suspiciously short extractions - likely a failed parse.
    if len(stripped_text) < minimum_characters:
        return False, f"too short ({len(stripped_text)} chars)"

    # Accept text that passes every check.
    return True, "ok"


# Test the validator against several realistic failure cases.
test_cases = [
    None,
    "",
    "   \n  ",
    "Short.",
    "Employees receive 24 annual leave days every year, per Section 7.1 of the policy.",
]

# Run each case and report the verdict.
for case in test_cases:
    is_valid, reason = validate_extracted_text(case)
    label = repr(case)[:45]
    print(f"  {'PASS' if is_valid else 'FAIL'}  {label:<48} {reason}")

## 14.2 Detecting extraction anomalies

Suppose average PDF text length is **20,000 characters**, and one document returns **17
characters**. That is suspicious — it could be a scanned document, an encrypted PDF, a
parser failure, or image-only content. A validation layer should catch it *before* it
silently enters the index as a useless chunk.

### The alphabetic ratio

Let $N_\alpha$ = alphabetic characters and $N$ = total non-whitespace characters:

$$R_\alpha = \frac{N_\alpha}{N}$$

If extracted text looks like `%$##@ 1| |||| 09 ?`, the alphabetic ratio is very low —
a strong signal of corruption or a failed OCR pass.

In [ ]:
# Define a function that computes the alphabetic character ratio.
def alphabetic_ratio(text):

    # Keep only non-whitespace characters - whitespace is not signal here.
    non_whitespace = [character for character in text if not character.isspace()]

    # An empty string has no ratio to compute.
    if not non_whitespace:
        return 0.0

    # Count how many of those characters are letters.
    alphabetic = sum(1 for character in non_whitespace if character.isalpha())

    # Ratio of letters to all visible characters.
    return alphabetic / len(non_whitespace)


# Clean prose should score high.
print(f"  clean text  : {alphabetic_ratio('Employees receive 24 annual leave days.'):.2f}")

# A numeric table scores much lower - which is normal for tables, not a failure.
print(f"  numeric data: {alphabetic_ratio('Q1: 12 | Q2: 17 | Q3: 19 | Q4: 21'):.2f}")

# Corrupted extraction scores very low - this is the alarm case.
print(f"  corrupted   : {alphabetic_ratio('%$##@ 1| |||| 09 ?'):.2f}")

> Note the middle case: a legitimate numeric table also scores low. So the alphabetic ratio
> is a **signal for review**, not an automatic reject. Every heuristic in this chapter needs
> that caveat — thresholds flag documents for a human, they do not make decisions alone.

## 14.3 Ingestion observability

Monitor: documents discovered · documents downloaded · parse success rate · OCR usage rate ·
OCR confidence · documents rejected · average document length · duplicates removed ·
processing latency · embedding failures · indexing failures.

An example dashboard:

```
Documents discovered    1,000,000
Parsed successfully       992,134
OCR fallback               42,110
Parse failures              1,420
Duplicate documents         6,446
Indexed                   984,268

Average parse time          0.82s
P95 parse time              4.70s
```

**That** is production RAG engineering — not *"the pipeline completed successfully."*

### Freshness lag

$$L = T_{\text{indexed}} - T_{\text{source update}}$$

If the source changed at 10:00 and the index updated at 17:00, then $L = 7\text{ hours}$.
If the business expects 10 minutes, your pipeline fails its **freshness SLA**.

| System | Acceptable lag |
|---|---|
| HR policies | 24 hours |
| Product inventory | seconds / minutes |
| Financial prices | milliseconds |
| Regulatory updates | hours |

This directly determines your architecture: batch vs event-driven vs live query.

## 14.4 Testing — the golden document set

Unit-test parsers against known documents: digital PDF · scanned PDF · multi-column PDF ·
PDF with table · password-protected PDF · empty PDF · DOCX headings · HTML boilerplate ·
Unicode content · very large document · duplicate document · updated document · deleted
document.

Build a manually validated set of **100–500 representative files**. For each, know the
expected title, page count, sections, tables, critical text and metadata. Then:

```
parser v1 → benchmark
parser v2 → benchmark
compare
```

This prevents **silent regressions** — the worst kind, because retrieval quality degrades
without any error being raised.

$$\text{Extraction Recall} = \frac{\text{Correctly extracted content}}{\text{Expected content}}$$

## 14.5 Benchmark criteria — more text is not better

When comparing parsers, do not ask *"which extracts more text?"*

| Metric | Why |
|---|---|
| Text accuracy | Correct characters |
| Reading order | Correct semantic sequence |
| Heading detection | Preserves hierarchy |
| Table accuracy | Maintains relationships |
| Image handling | Multimodal support |
| OCR quality | Scan support |
| Speed | Throughput |
| Memory | Scale |
| Cost | Production viability |
| Metadata | Provenance |

**Parser A** extracts 10,000 characters — but includes menus, headers, footers, navigation
and duplicated text.
**Parser B** extracts 7,000 characters — accurately capturing only main content.

**Parser B produces much better RAG.**

$$\text{Quantity} \neq \text{Quality}$$

# 15. Putting it together — a complete mini pipeline

Each function has exactly one responsibility. The orchestrator calls them in sequence.

```
extract_text()
      ↓
clean_unicode()
      ↓
clean_spacing()
      ↓
is_valid_document()
      ↓
create_content_hash()
      ↓
create_canonical_document()
```

In [ ]:
# ---- Step 1: extract -------------------------------------------------------

# Define a function that extracts text from a plain text file.
def extract_text_file(file_path):

    # Open the file with explicit UTF-8 encoding.
    with open(file_path, "r", encoding="utf-8") as file:

        # Read the entire content into memory.
        text = file.read()

    # Return the raw extracted text.
    return text


# ---- Step 2: normalize Unicode --------------------------------------------

# Define a function that normalizes Unicode to canonical composed form.
def clean_unicode(text):

    # NFC composes decomposed characters into their single-code-point equivalents.
    return unicodedata.normalize("NFC", text)


# ---- Step 3: normalize spacing --------------------------------------------

# Define a function that cleans horizontal whitespace while keeping line structure.
def clean_spacing(text):

    # Collapse repeated spaces and tabs inside each line.
    cleaned_text = re.sub(r"[ \t]+", " ", text)

    # Remove leading and trailing whitespace from the whole document.
    cleaned_text = cleaned_text.strip()

    # Return the cleaned text.
    return cleaned_text


# ---- Step 4: validate ------------------------------------------------------

# Define a function that decides whether a document is worth indexing.
def is_valid_document(text):

    # Reject missing text.
    if text is None:
        return False

    # Reject whitespace-only text.
    if not text.strip():
        return False

    # Reject suspiciously small documents.
    if len(text.strip()) < 50:
        return False

    # Accept everything else.
    return True


# ---- Step 5: hash ----------------------------------------------------------

# Define a function that produces a stable content checksum.
def create_content_hash(text):

    # Encode to bytes, hash with SHA-256, render as hex.
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


# ---- Step 6: build the canonical document ---------------------------------

# Define a function that assembles the final canonical Document object.
def create_canonical_document(document_id, raw_text, clean_text, metadata, checksum):

    # Construct the dataclass instance defined in section 2.
    return Document(
        document_id=document_id,
        text=clean_text,
        metadata=metadata,
        raw_text=raw_text,
        checksum=checksum,
    )

In [ ]:
# ---- The orchestrator ------------------------------------------------------

# Define the end-to-end ingestion of a single text file.
def ingest_text_file(file_path, document_id, metadata):

    # Extract the raw text from disk.
    raw_text = extract_text_file(file_path)

    # Normalize Unicode representation.
    unicode_cleaned = clean_unicode(raw_text)

    # Normalize whitespace.
    clean_text = clean_spacing(unicode_cleaned)

    # Stop early if the extraction is unusable - fail loudly, not silently.
    if not is_valid_document(clean_text):
        raise ValueError(f"Document {document_id} failed validation.")

    # Compute the checksum over the CLEAN text, so cosmetic changes do not
    # trigger unnecessary re-embedding.
    checksum = create_content_hash(clean_text)

    # Assemble and return the canonical document.
    return create_canonical_document(
        document_id=document_id,
        raw_text=raw_text,
        clean_text=clean_text,
        metadata=metadata,
        checksum=checksum,
    )


# Write a sample policy file so the pipeline has real input.
sample_txt = DATA_DIR / "policy.txt"

# Include deliberate messiness: extra spaces and a decomposed Unicode character.
sample_txt.write_text(
    "Employee   Policy  2026\n\n"
    "Employees receive 24 annual leave days every year.\n"
    "The office in Zürich follows local holidays.\n",
    encoding="utf-8",
)

# Run the full pipeline.
canonical = ingest_text_file(
    sample_txt,
    document_id="local:policy_001",
    metadata={"source": "local_file", "file_name": "policy.txt", "department": "HR"},
)

# Inspect the result.
print("document_id :", canonical.document_id)
print("checksum    :", canonical.checksum[:16], "...")
print("metadata    :", canonical.metadata)
print("clean text  :")
print(canonical.text)

In [ ]:
# Demonstrate idempotency: running the same ingestion twice must produce the same checksum.
second_run = ingest_text_file(
    sample_txt,
    document_id="local:policy_001",
    metadata={"source": "local_file", "file_name": "policy.txt", "department": "HR"},
)

# Same input, same stable ID, same checksum - so an UPSERT is a no-op.
print("run 1 checksum:", canonical.checksum[:16])
print("run 2 checksum:", second_run.checksum[:16])
print("idempotent?   ", canonical.checksum == second_run.checksum)

# 16. Anti-patterns

| # | Anti-pattern | Why it hurts |
|---|---|---|
| 1 | `PDF → extract_text() → split every 1000 chars → vector DB` | Fine for demos, risky for production |
| 2 | Discarding all metadata | No filtering, no citations, no access control |
| 3 | Using OCR on every PDF | Enormous unnecessary cost |
| 4 | Reprocessing every document after every update | 99% wasted compute |
| 5 | Ignoring deleted documents | Correctness *and* security failure |
| 6 | Random IDs on every ingestion run | Guaranteed duplicates |
| 7 | Cleaning aggressively without retaining raw text | Unrecoverable information loss |
| 8 | Ignoring ACLs until after indexing | Data leak baked into the index |
| 9 | Flattening tables into meaningless text | Column relationships destroyed |
| 10 | Treating every source with the same parser | Wrong tool, silent quality loss |


# 17. The RAG debugging ladder (extended)

Chapter 1 gave you retrieval-vs-generation. Chapter 2 adds the two stages **above** them:

```
Wrong answer
    │
    ▼
Does the source contain the correct answer?
    ├── NO  → source-data issue
    └── YES
         │
         ▼
    Was the answer parsed correctly?
         ├── NO  → INGESTION issue        ← Chapter 2
         └── YES
              │
              ▼
         Was it chunked properly?
              ├── NO  → chunking issue    ← Chapter 3
              └── YES
                   │
                   ▼
              Was the chunk retrieved?
                   ├── NO  → retrieval issue
                   └── YES
                        │
                        ▼
                   Did the reranker keep it?
                        ├── NO  → ranking issue
                        └── YES
                             │
                             ▼
                        Did the LLM answer correctly?
                             ├── NO  → generation issue
                             └── YES → not actually a bug
```

### Ingestion failure vs chunking failure — tell them apart

Original source:

```
Section 7
Employees with 5+ years receive 30 days leave.
```

| Parser output | Diagnosis |
|---|---|
| `Section 7\nEmployees with 5+ years receive 30 days leave.` but badly split later | **Chunking failure** |
| `Section 7\nEmployees with 5+` — content missing | **Ingestion / parsing failure** |

Debugging depends entirely on identifying the correct stage. Guessing wastes days.

## Ingestion failure taxonomy

```
                  INGESTION FAILURES
                          │
      ┌───────────────────┼───────────────────┐
      │                   │                   │
   Parsing            Structure            Metadata
      │                   │                   │
  empty text         wrong order         missing ACL
  OCR errors         lost tables         wrong dates
  corrupt text       lost headings       missing source
      │                   │                   │
      └───────────────────┼───────────────────┘
                          │
                      Freshness
                          │
                     old versions
                     stale indexes
                    deletions missed
```

# 18. Interview preparation

### Q: "Why is PDF parsing difficult?"

> PDF is a presentation-oriented format. It often stores text as positioned glyphs rather
> than explicit semantic structures such as paragraphs and tables. Multi-column layouts,
> scanned pages, headers, footers, tables, figures, and reading-order ambiguity make
> extraction difficult. Production RAG therefore often requires layout-aware parsers, OCR
> fallback, structural reconstruction, and extraction-quality validation.

### Q: "How do you ingest a scanned PDF?"

> 1. Detect insufficient embedded text
> 2. Render page to image
> 3. Preprocess image if necessary (deskew, denoise, contrast)
> 4. Run OCR
> 5. Capture words + bounding boxes + confidence
> 6. Perform layout analysis
> 7. Reconstruct reading order
> 8. Extract headings / tables / paragraphs
> 9. Validate OCR quality
> 10. Preserve page-level provenance

### Q: "How would you avoid indexing duplicates?"

> Normalize content → SHA-256 for exact duplicates → stable source IDs → version-aware
> upsert → near-duplicate detection for similar documents. At scale, use MinHash/LSH rather
> than O(N²) pairwise comparison.

### Q: "How do you handle updated documents?"

> Compare source ID plus updated timestamp / ETag / content hash against the stored version.
> If changed: reparse → rechunk → re-embed **only the affected document** → replace old
> chunks atomically, building the new version before switching over so the document never
> disappears from search.

### Q: "Where do permissions belong?"

> Permissions should be acquired from the source during ingestion, stored as metadata
> associated with documents and chunks, and enforced by retrieval filters before
> unauthorized content enters the LLM context. Prompt-level instructions are not a security
> boundary.

### Q: "How do you know ingestion is working?"

> Parse success rate, OCR confidence, empty-extraction rate, average content length,
> duplicate rate, freshness lag, indexing failures, document counts, metadata completeness,
> and sample-based quality evaluation against a golden document set — **not** merely
> *"the pipeline completed successfully."*

### Q: "What should be stored for citations?"

> At minimum: `document_id`, file name/title, page/section, source URL, chunk offset,
> version. So the answer can cite *"Employee Handbook, Section 7.2, Page 31."*

### Q: "Design ingestion for 10 million enterprise documents across SharePoint, S3, Confluence and databases."

```
                  CONNECTORS
        ┌──────────────┼──────────────┐
   SharePoint         S3         Confluence
        └──────────────┼──────────────┘
                       ▼
                   Discovery
                       │
                       ▼
               Metadata Database
                       │
                       ▼
                  Event Queue
                       │
        ┌──────────────┼──────────────┐
        ▼              ▼              ▼
    Worker 1       Worker 2       Worker N
                       │
                       ▼
                 Parser Router
                       │
        ├── PDF parser     ├── HTML parser
        ├── OCR service    └── DOCX parser
                       │
                       ▼
           Cleaning + normalization
                       │
                       ▼
            Metadata / ACL enrichment
                       │
                       ▼
            Deduplication + validation
                       │
                       ▼
            Canonical Document Store
                       │
                       ▼
              Chunking Pipeline
                       │
                       ▼
              Embedding Workers
                       │
                       ▼
            Vector / Search Index
```

Then add: **DLQ · retry · metrics · versioning · incremental updates · deletion processing
· idempotency** — and the answer becomes production-ready.

### The one-paragraph summary answer

> I separate ingestion into connectors, raw storage, parser routing, structure extraction,
> cleaning and normalization, metadata and ACL enrichment, deduplication/versioning,
> validation, and canonical storage. PDFs may require digital-text extraction or OCR/layout
> parsing depending on their type. I preserve provenance and structural information such as
> sections, pages, tables, and headings because downstream chunking and citations depend on
> it. The pipeline should be incremental, idempotent, observable, retryable, and capable of
> propagating updates and deletions into the retrieval index.

## Your turn — answer from memory

1. A user reports a wrong answer. The source PDF definitely contains the right text. Walk through your diagnosis.
2. Why keep canonical parsed documents in a separate store from the vector DB?
3. Your OCR bill is \$50K/month. What do you check first?
4. How do you make an ingestion worker idempotent, and why does it matter?
5. Why is `document_id = uuid4()` a bug?
6. When is aggressive text cleaning actively harmful? Give three examples.

# 19. Chapter summary

## The 12 key principles

1. **Garbage In → Garbage Out.** RAG quality begins with data quality.
2. PDF extraction is **not** trivial text reading.
3. **Preserve structure:** headings · sections · tables · pages · figures.
4. **Keep metadata.**
5. **Keep provenance.**
6. **Carry ACL/security metadata** during ingestion, not after.
7. **Maintain stable document IDs.**
8. **Deduplicate** documents.
9. **Handle versions, updates and deletions.**
10. **Keep original/raw content** where feasible.
11. Use **incremental** rather than complete re-indexing.
12. **Do not flatten valuable structure** before chunking.

## How Chapters 1 and 2 connect

Chapter 1 said:

```
Documents → Chunk → Embed → Retrieve → Generate
```

You should now mentally expand `Documents` into:

```
Source → Connector → Parse → OCR/Layout → Clean → Normalize
       → Metadata → ACL → Deduplicate → Version → Validate
       → Canonical Document → CHUNK
```

## Key formulas

| Formula | Meaning |
|---|---|
| $Q_{\text{RAG}} \approx Q_{\text{ing}} \times Q_{\text{ret}} \times Q_{\text{gen}}$ | Ingestion caps everything downstream |
| $C_{\text{page}} = \frac{1}{N}\sum c_i$ | Average OCR confidence |
| $f(l) = \frac{\#\text{pages with } l}{N}$ | Boilerplate detection frequency |
| $J(A,B) = \frac{\lvert A \cap B \rvert}{\lvert A \cup B \rvert}$ | Jaccard near-duplicate similarity |
| $R_\alpha = N_\alpha / N$ | Alphabetic ratio (corruption signal) |
| $\text{delay}_n = \min(d_{\max}, d_0 \cdot 2^n)$ | Exponential backoff |
| $L = T_{\text{indexed}} - T_{\text{source}}$ | Freshness lag |
| $f(f(x)) = f(x)$ | Idempotency |


## Exercises

1. **Break the boilerplate detector.** Create a 5-page PDF where one page *legitimately*
   repeats a critical condition. Watch `detect_repeated_lines` flag it. How would you fix
   the heuristic?
2. **Build a parser benchmark.** Extract the same PDF with PyMuPDF and pdfplumber. Compare
   character count, reading order and table fidelity. Which "extracted more"? Which was better?
3. **Implement near-duplicate clustering.** Given 20 documents, group them by Jaccard
   similarity > 0.8. Then measure how the runtime scales as you add documents.
4. **Simulate a deletion bug.** Ingest 3 documents, delete one at the source, re-run ingestion
   *without* deletion handling. Confirm the deleted content is still retrievable.
5. **Add page-number stripping.** Extend the cleaner to remove `Page N of M` using a regex,
   and verify it does not damage a sentence that legitimately contains the word "page".
6. **Design a golden set.** Pick 5 real PDFs. Write down expected title, page count, section
   headings and one critical sentence for each. That is the seed of a regression test suite.


## → Next: Chapter 3 — Document Chunking

Now that we have clean, structured, metadata-rich canonical documents, we face the question
that determines retrieval quality more than any other single choice:

> **How do we split a document into retrievable units?**

Chapter 3 covers: why chunk at all · what exactly is a chunk · how large · what is overlap
and why it can hurt · characters vs words vs tokens · fixed-size · sentence · paragraph ·
recursive character splitting · token-based · structure-aware · Markdown/header ·
semantic · sliding-window · parent-child · small-to-big retrieval · contextual chunking ·
chunking tables, code and legal documents · how chunk size affects embedding similarity and
Recall@K · and how to choose chunk size both mathematically and experimentally.